In [1]:
from pathlib import Path
import os


def _human_size(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024


def _dir_size_bytes(path: Path) -> int:
    total = 0
    if not path.exists():
        return 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total


hf_home = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
hub_dir = hf_home / "hub"

total_bytes = _dir_size_bytes(hf_home)
hub_bytes = _dir_size_bytes(hub_dir)

print(f"HF_HOME: {hf_home}")
print(f"Tổng cache Hugging Face: {_human_size(total_bytes)} ({total_bytes:,} bytes)")
print(f"Riêng thư mục hub: {_human_size(hub_bytes)} ({hub_bytes:,} bytes)")

if hub_dir.exists():
    repo_dirs = [p for p in hub_dir.iterdir() if p.is_dir()]
    top = []
    for d in repo_dirs:
        size_b = _dir_size_bytes(d)
        top.append((size_b, d.name))
    top.sort(reverse=True)

    print("\nTop 10 repo cache nặng nhất:")
    for idx, (size_b, name) in enumerate(top[:10], start=1):
        print(f"{idx:02d}. {name:<60} {_human_size(size_b)}")
else:
    print("Không tìm thấy thư mục hub cache.")

HF_HOME: C:\Users\luong\.cache\huggingface
Tổng cache Hugging Face: 1.76 GB (1,887,446,117 bytes)
Riêng thư mục hub: 1.24 GB (1,331,691,560 bytes)

Top 10 repo cache nặng nhất:
01. datasets--McAuley-Lab--Amazon-Reviews-2023                   1.24 GB
02. .locks                                                       0.00 B


In [2]:
from huggingface_hub import scan_cache_dir

cache = scan_cache_dir()
total_before = cache.size_on_disk

print(f"Cache hiện tại trước khi xóa: {total_before / (1024**3):.2f} GB")

all_revision_hashes = [
    rev.commit_hash
    for repo in cache.repos
    for rev in repo.revisions
]

strategy = cache.delete_revisions(*all_revision_hashes)
print(f"Dự kiến giải phóng: {strategy.expected_freed_size / (1024**3):.2f} GB")

strategy.execute()

cache_after = scan_cache_dir()
freed = total_before - cache_after.size_on_disk

print(f"Đã xóa cache Hugging Face. Giải phóng: {freed / (1024**3):.2f} GB")
print(f"Cache còn lại: {cache_after.size_on_disk / (1024**3):.2f} GB")

c:\Users\luong\miniconda3\envs\data\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\luong\miniconda3\envs\data\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cache hiện tại trước khi xóa: 0.00 GB
Dự kiến giải phóng: 0.00 GB
Đã xóa cache Hugging Face. Giải phóng: 0.00 GB
Cache còn lại: 0.00 GB
